# Example: Stress-Testing the Minimum-Variance Portfolio

In this example, we run systematic "what-if" experiments on the baseline minimum-variance portfolio from the previous example. We'll shift correlations, simulate price drops, and increase trading costs — then produce a performance scorecard that summarizes how the portfolio degrades under stress.

> **By the end of this example, you will be able to:**
> * Implement correlation stress tests by blending toward a crisis-regime correlation matrix
> * Simulate price drop scenarios and compute portfolio-level impact
> * Evaluate the effect of increased trading costs on portfolio performance
> * Produce a comprehensive baseline scorecard (return, drawdown, turnover, trading cost)

## Setup, Data and Prerequisites
Load our packages and the baseline portfolio from the previous example.

In [ ]:
include("Include.jl");

Load the baseline portfolio and synthetic return data generated in the previous example.

In [ ]:
let
    # Load baseline portfolio and synthetic data -
    portfolio_data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    returns_data = load(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"));

    # Unpack into global-scope constants for use in later cells -
    global baseline_weights = portfolio_data["weights"];
    global μ_hat = portfolio_data["mu_hat"];
    global Σ_hat = portfolio_data["Sigma_hat"];
    global asset_names = portfolio_data["asset_names"];
    global df_returns = returns_data["returns"];
    global C_true = returns_data["C_true"];
    global N = length(asset_names);
    global R_target = 0.05 / 252;
    global bounds = hcat(zeros(N), ones(N));

    println("Loaded baseline portfolio with $(N) assets: $(asset_names)")
    println("Baseline weights: $(round.(baseline_weights .* 100, digits=1))%")
end

___
## Task 1: Stress Test — Correlation Shifts
We perturb the correlation structure toward a "crisis regime" where all assets become more correlated. Using the blending formula $\tilde{\mathbf{C}} = (1-\alpha)\mathbf{C} + \alpha\,\mathbf{1}\mathbf{1}^{\top}$, we test $\alpha \in \{0.0, 0.10, 0.25, 0.50\}$ and re-solve the optimization under each stressed covariance matrix.

> **What should you see?** As correlations increase, the diversification benefit shrinks. Portfolio variance _increases_ even though the optimizer is doing its best. The weights will also shift — the Bond becomes even more dominant because it's the only asset offering genuine diversification (negative correlation with equities).

In [ ]:
let
    # Extract the estimated standard deviations and correlation from Σ_hat -
    σ_hat = sqrt.(diag(Σ_hat));
    D_inv = diagm(1.0 ./ σ_hat);
    C_hat = D_inv * Σ_hat * D_inv; # estimated correlation matrix
    D_mat = diagm(σ_hat);

    # Stress levels -
    αs = [0.0, 0.10, 0.25, 0.50];
    ones_matrix = ones(N, N);

    # Results storage -
    stress_results = DataFrame();

    for α in αs
        
        # Blend correlation toward crisis -
        C_stressed = (1 - α) .* C_hat .+ α .* ones_matrix;

        # Rebuild covariance matrix -
        Σ_stressed = D_mat * C_stressed * D_mat;

        # Solve -
        problem = build(MyPortfolioAllocationProblem;
            μ = μ_hat, Σ = Σ_stressed, bounds = bounds, R = R_target);
        result = solve_minvariance(problem);

        # Portfolio volatility under stress -
        vol_annual = round(sqrt(result.variance) * sqrt(252) * 100, digits=2);
        turnover = round(compute_turnover(baseline_weights, result.weights), digits=4);

        row = DataFrame(
            "α" => α,
            [name => round(w * 100, digits=1) for (name, w) in zip(asset_names, result.weights)]...,
            "Vol (%)" => vol_annual,
            "Turnover" => turnover
        );
        stress_results = vcat(stress_results, row);
    end

    println("Correlation Stress Test Results:")
    pretty_table(stress_results, tf=tf_markdown)
end

___
## Task 2: Stress Test — Price Drops
We simulate sudden price drops in individual assets and measure the impact on portfolio value. For each asset, we apply a one-day shock of -10%, -20%, and -40% and compute the portfolio-level loss.

> **What should you see?** The portfolio's sensitivity to each shock depends on the asset's weight. A 40% drop in a heavily-weighted asset (likely Bond) is devastating, while a 40% drop in a near-zero-weight asset barely registers. This reveals the _hidden concentration risk_ in the min-variance solution.

In [ ]:
let
    # Price drop scenarios -
    drop_levels = [-0.10, -0.20, -0.40];
    
    # Results -
    drop_results = DataFrame();

    for i in 1:N
        for drop in drop_levels
            
            # Portfolio loss = weight_i × drop
            portfolio_loss = baseline_weights[i] * drop * 100;

            row = DataFrame(
                "Asset" => asset_names[i],
                "Weight (%)" => round(baseline_weights[i] * 100, digits=1),
                "Drop (%)" => Int(drop * 100),
                "Portfolio Loss (%)" => round(portfolio_loss, digits=2)
            );
            drop_results = vcat(drop_results, row);
        end
    end

    println("Price Drop Stress Test (portfolio-level impact):")
    pretty_table(drop_results, tf=tf_markdown)
end

**Visualize:** Heatmap of portfolio losses across assets and drop levels.

> **What should you see?** A heatmap where the color intensity tracks the portfolio-level loss. The row corresponding to the highest-weight asset will show the deepest red — that's where the concentration risk lives.

In [ ]:
let
    # Build loss matrix: rows = assets, cols = drop levels -
    drop_levels = [0.10, 0.20, 0.40];
    loss_matrix = zeros(N, length(drop_levels));
    
    for (j, drop) in enumerate(drop_levels)
        for i in 1:N
            loss_matrix[i, j] = baseline_weights[i] * drop * 100;
        end
    end

    heatmap(["−10%", "−20%", "−40%"], asset_names, loss_matrix,
        xlabel="Price Drop", ylabel="Asset", title="Portfolio Loss (%) by Asset Drop",
        color=:Reds, size=(600, 400), clims=(0, maximum(loss_matrix)))
end

___
## Task 3: Trading Cost Stress Test and Baseline Scorecard
We evaluate how portfolio performance changes as trading costs increase, then assemble a comprehensive scorecard that summarizes the baseline min-variance portfolio's performance. This scorecard will serve as the benchmark throughout the remaining sessions.

> **What should you see?** High-turnover portfolios suffer most from cost increases. The scorecard captures expected return, volatility, maximum drawdown, turnover, and trading cost — the numbers the AI rebalancing engine in Session 2 needs to beat.

In [ ]:
let
    # Baseline cost assumption: 10 bps per unit traded -
    base_cost_bps = 10.0;
    cost_multipliers = [1, 2, 5, 10];
    
    # Starting point: equal-weight portfolio -
    w_equal = fill(1.0 / N, N);
    turnover = compute_turnover(w_equal, baseline_weights);

    # Results -
    cost_results = DataFrame();

    for mult in cost_multipliers
        cost_bps = base_cost_bps * mult;
        trading_cost = turnover * (cost_bps / 10000) * 100; # as % of portfolio

        row = DataFrame(
            "Cost Multiplier" => "$(mult)×",
            "Cost (bps/trade)" => Int(cost_bps),
            "Turnover" => round(turnover, digits=4),
            "Trading Cost (%)" => round(trading_cost, digits=4)
        );
        cost_results = vcat(cost_results, row);
    end

    println("Trading Cost Stress Test (equal-weight → min-variance rebalance):")
    pretty_table(cost_results, tf=tf_markdown)
end

**Baseline Scorecard:** Now we assemble the comprehensive scorecard from the synthetic return data.

> **What should you see?** A single table summarizing: expected return, portfolio volatility, maximum drawdown, turnover relative to equal-weight, and trading cost.

In [ ]:
let
    # Compute portfolio returns over the synthetic data period -
    R = Matrix(df_returns); # T × N
    portfolio_returns = R * baseline_weights; # T × 1

    # Scorecard metrics -
    expected_return_annual = mean(portfolio_returns) * 252 * 100;
    volatility_annual = std(portfolio_returns) * sqrt(252) * 100;
    max_drawdown = compute_drawdown(portfolio_returns) * 100;
    
    w_equal = fill(1.0 / N, N);
    turnover = compute_turnover(w_equal, baseline_weights);
    trading_cost = turnover * (10.0 / 10000) * 100; # 10 bps baseline

    # Build the scorecard -
    scorecard = DataFrame(
        "Metric" => [
            "Expected Return (annual %)",
            "Volatility (annual %)",
            "Sharpe Ratio (approx)",
            "Maximum Drawdown (%)",
            "Turnover (vs equal-weight)",
            "Trading Cost (%, at 10 bps)"
        ],
        "Value" => [
            round(expected_return_annual, digits=2),
            round(volatility_annual, digits=2),
            round(expected_return_annual / volatility_annual, digits=2),
            round(max_drawdown, digits=2),
            round(turnover, digits=4),
            round(trading_cost, digits=4)
        ]
    );

    println("═"^50)
    println("  BASELINE SCORECARD: Min-Variance Portfolio")
    println("═"^50)
    pretty_table(scorecard, tf=tf_markdown)

    # Save the scorecard for use in later sessions -
    save(joinpath(_PATH_TO_DATA, "baseline-scorecard.jld2"),
        Dict("scorecard" => scorecard, "portfolio_returns" => portfolio_returns,
             "baseline_weights" => baseline_weights));
end

**Visualize:** Cumulative wealth of the min-variance portfolio vs. an equal-weight benchmark over the synthetic data period.

> **What should you see?** Two wealth curves starting at $1.00. The min-variance portfolio should show lower volatility (smoother path) but potentially lower total return than the equal-weight portfolio. The drawdown periods — where the curve dips from its peak — show the portfolio's vulnerability.

In [ ]:
let
    # Compute cumulative wealth for both portfolios -
    R = Matrix(df_returns);
    
    # Min-variance portfolio -
    mv_returns = R * baseline_weights;
    mv_wealth = cumprod(1.0 .+ mv_returns);

    # Equal-weight benchmark -
    w_equal = fill(1.0 / N, N);
    ew_returns = R * w_equal;
    ew_wealth = cumprod(1.0 .+ ew_returns);

    # Plot -
    T = length(mv_wealth);
    days = 1:T;

    plot(days, mv_wealth, label="Min-Variance", linewidth=2, color=:steelblue)
    plot!(days, ew_wealth, label="Equal-Weight", linewidth=2, color=:coral, linestyle=:dash)
    xlabel!("Trading Day")
    ylabel!("Cumulative Wealth (\$1 initial)")
    title!("Min-Variance vs. Equal-Weight Portfolio")
    plot!(size=(700, 400), legend=:topleft)
end

___
## Summary
In this example, we stress-tested the classical minimum-variance portfolio across three dimensions — correlation spikes, price drops, and trading cost increases — and produced a baseline performance scorecard. The key findings:

- **Correlation stress** increases portfolio volatility and further concentrates weights in the Bond
- **Price drop exposure** is dominated by the highest-weight asset — concentration risk in action
- **Trading costs** scale linearly with turnover, penalizing portfolios that deviate far from simpler allocations
- **The baseline scorecard** provides the quantitative benchmark that the AI rebalancing engine (Session 2) must beat

The scorecard and portfolio data are saved to the `data/` directory for use in subsequent sessions.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.